# Projektstruktur-Analyse: edu-code-course-ml

Dieses Notebook analysiert und vervollständigt automatisch die Verzeichnisstruktur des Projekts.

**Soll-Struktur:**
```
edu-code-course-ml/
├── template/         ← Submodule: edu-code-projecttemplate
├── informationen/    ← Hilfsmittel für Schüler
├── aufgaben/         ← ML-Aufgaben für Schüler
├── loesungen/        ← Musterlösungen
├── notebooks/        ← Jupyter Notebooks
├── tests/            ← Tests zu den Aufgaben
└── README.md
```

## 1. Erforderliche Bibliotheken importieren

In [ ]:
import os
import subprocess
from pathlib import Path

# Projektwurzel (ein Verzeichnis über dem notebooks/-Ordner)
PROJEKT_ROOT = Path("..").resolve()
print(f"Projektwurzel: {PROJEKT_ROOT}")

## 2. Projektstruktur analysieren

Liest alle vorhandenen Verzeichnisse und Dateien rekursiv aus.

In [ ]:
IGNORIEREN = {".git", "__pycache__", ".ipynb_checkpoints"}

print(f"Aktuelle Struktur von: {PROJEKT_ROOT.name}/\n")

for pfad in sorted(PROJEKT_ROOT.rglob("*")):
    # .git und andere interne Ordner überspringen
    if any(teil in pfad.parts for teil in IGNORIEREN):
        continue
    # Einrückungstiefe berechnen
    tiefe = len(pfad.relative_to(PROJEKT_ROOT).parts) - 1
    einzug = "    " * tiefe
    symbol = "├── "
    if pfad.is_dir():
        print(f"{einzug}{symbol}{pfad.name}/")
    else:
        print(f"{einzug}{symbol}{pfad.name}")

## 3. Fehlende Verzeichnisse und Dateien identifizieren

Vergleich zwischen Ist- und Soll-Struktur.

In [ ]:
# Soll-Struktur: Verzeichnisse und Pflichtdateien
SOLL_VERZEICHNISSE = [
    "template",
    "informationen",
    "aufgaben",
    "loesungen",
    "notebooks",
    "tests",
]

SOLL_DATEIEN = [
    "README.md",
    "aufgaben/README.md",
    "loesungen/README.md",
    "notebooks/README.md",
    "tests/README.md",
    "informationen/README.md",
]

fehlende_verzeichnisse = []
fehlende_dateien = []

for verz in SOLL_VERZEICHNISSE:
    pfad = PROJEKT_ROOT / verz
    if not pfad.exists():
        fehlende_verzeichnisse.append(verz)

for datei in SOLL_DATEIEN:
    pfad = PROJEKT_ROOT / datei
    if not pfad.exists():
        fehlende_dateien.append(datei)

print("=== Fehlende Verzeichnisse ===")
if fehlende_verzeichnisse:
    for v in fehlende_verzeichnisse:
        print(f"  ✗ {v}/")
else:
    print("  ✓ Alle Pflichtverzeichnisse vorhanden")

print("\n=== Fehlende Pflichtdateien ===")
if fehlende_dateien:
    for d in fehlende_dateien:
        print(f"  ✗ {d}")
else:
    print("  ✓ Alle Pflichtdateien vorhanden")

## 4. Fehlende Bestandteile erstellen

Fehlende Verzeichnisse und Pflichtdateien werden automatisch angelegt.

In [ ]:
erstellt = []

# Fehlende Verzeichnisse anlegen
for verz in fehlende_verzeichnisse:
    pfad = PROJEKT_ROOT / verz
    pfad.mkdir(parents=True, exist_ok=True)
    # .gitkeep anlegen damit das Verzeichnis in Git getrackt wird
    (pfad / ".gitkeep").touch()
    erstellt.append(f"Verzeichnis: {verz}/")
    print(f"  ✓ Verzeichnis angelegt: {verz}/")

# Fehlende Pflichtdateien anlegen
for datei in fehlende_dateien:
    pfad = PROJEKT_ROOT / datei
    pfad.parent.mkdir(parents=True, exist_ok=True)
    if not pfad.exists():
        pfad.write_text(f"# {pfad.parent.name.capitalize()}\n\nInhalt folgt.\n", encoding="utf-8")
        erstellt.append(f"Datei: {datei}")
        print(f"  ✓ Datei angelegt: {datei}")

if not erstellt:
    print("  ✓ Keine Aktion notwendig – Struktur bereits vollständig.")
else:
    print(f"\nInsgesamt {len(erstellt)} Elemente erstellt.")

## 5. Submodul-Konfiguration prüfen

Prüft, ob `edu-code-projecttemplate` korrekt als Git-Submodul eingebunden ist.

In [ ]:
gitmodules_pfad = PROJEKT_ROOT / ".gitmodules"

print("=== .gitmodules Inhalt ===")
if gitmodules_pfad.exists():
    print(gitmodules_pfad.read_text(encoding="utf-8"))
else:
    print("  ✗ .gitmodules nicht gefunden – Submodul nicht eingebunden!")

print("=== Git Submodul Status ===")
ergebnis = subprocess.run(
    ["git", "submodule", "status"],
    cwd=PROJEKT_ROOT,
    capture_output=True,
    text=True
)
if ergebnis.stdout.strip():
    for zeile in ergebnis.stdout.strip().splitlines():
        status_zeichen = zeile[0]
        status_text = {
            " ": "✓ Korrekt eingebunden",
            "-": "✗ Nicht initialisiert – führe 'git submodule update --init' aus",
            "+": "⚠ Neuere Version verfügbar",
            "U": "✗ Merge-Konflikt"
        }.get(status_zeichen, "? Unbekannter Status")
        print(f"  {status_text}: {zeile[1:].strip()}")
else:
    print("  ✗ Keine Submodule gefunden")

## 6. Struktur validieren

Abschließender Statusbericht: Ist- vs. Soll-Zustand.

In [ ]:
print("=" * 50)
print("  VALIDIERUNGSBERICHT: Projektstruktur")
print("=" * 50)

alle_ok = True

print("\n[ Verzeichnisse ]")
for verz in SOLL_VERZEICHNISSE:
    pfad = PROJEKT_ROOT / verz
    if pfad.exists() and pfad.is_dir():
        print(f"  ✓  {verz}/")
    else:
        print(f"  ✗  {verz}/   ← FEHLT")
        alle_ok = False

print("\n[ Pflichtdateien ]")
for datei in SOLL_DATEIEN:
    pfad = PROJEKT_ROOT / datei
    if pfad.exists():
        print(f"  ✓  {datei}")
    else:
        print(f"  ✗  {datei}   ← FEHLT")
        alle_ok = False

print("\n[ Submodul ]")
submodul_pfad = PROJEKT_ROOT / "template"
if submodul_pfad.exists() and any(submodul_pfad.iterdir()):
    print("  ✓  template/ (Submodul eingebunden und befüllt)")
elif submodul_pfad.exists():
    print("  ⚠  template/ existiert, aber ist leer → git submodule update --init --recursive")
    alle_ok = False
else:
    print("  ✗  template/ fehlt")
    alle_ok = False

print("\n" + "=" * 50)
if alle_ok:
    print("  ERGEBNIS: ✓ Struktur vollständig und korrekt")
else:
    print("  ERGEBNIS: ✗ Struktur unvollständig – siehe Punkte oben")
print("=" * 50)